# Data preparation on the Microbiology dataset

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))
from src.utils import data_understanding as du

# Get the notebook directory and construct absolute path
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
csv_path = os.path.join(notebook_dir, "../../data/raw/microbiology_events_codes_3.csv")
csv_path = os.path.normpath(csv_path)

print(f"Looking for file at: {csv_path}")
print(f"File exists: {os.path.exists(csv_path)}")

microBio_df = pd.read_csv(csv_path)

print(microBio_df['subject_id'].nunique())
microBio_df

In [ ]:
# Drop the technician_id column
microBio_df = microBio_df.drop(columns=['technician_id'])
microBio_df

In [ ]:
# Drops rows where no organism has been detected
#microBio_df = microBio_df[~microBio_df['org_name'].isna()]
#microBio_df

In [ ]:
# Drops duplicates
microBio_df = microBio_df.drop_duplicates()
microBio_df

In [ ]:
# Drops rows where the two times does not correspond
chartdates = pd.to_datetime(microBio_df['chartdate'])
charttimes = pd.to_datetime(microBio_df['charttime'])

mask = chartdates.dt.date == charttimes.dt.date
microBio_df = microBio_df[mask]

microBio_df

In [ ]:
# Drops the rows where dilution_comparison + dilution_value are different from dilution_text
# Creates the string to compare
def format_val(x):
    try:
        x = float(x)
    except (ValueError, TypeError):
        return ""
    return str(int(x)) if x.is_integer() else str(x)

comp = microBio_df['dilution_comparison'].str.strip()
val = microBio_df['dilution_value'].apply(format_val)

dilution_constructed = np.where(
    comp == '=',
    val,
    (comp + val).str.replace(" ", "")
)

match_mask = microBio_df['dilution_text'].fillna('') == pd.Series(
    dilution_constructed, index=microBio_df.index
).fillna('')

dropped_rows = microBio_df[~match_mask].copy()
dropped_rows

microBio_df = microBio_df[match_mask]


microBio_df



In [ ]:
# Standardizes texts
microBio_df = microBio_df.copy()
microBio_df['spec_type_desc'] = microBio_df['spec_type_desc'].str.strip().str.upper()
microBio_df['test_name'] = microBio_df['test_name'].str.strip().str.upper()
microBio_df['org_name'] = microBio_df['org_name'].str.strip().str.upper()
microBio_df['ab_name'] = microBio_df['ab_name'].str.strip().str.upper()
microBio_df['interpretation'] = microBio_df['interpretation'].str.strip().str.upper()
microBio_df['qc_flag'] = microBio_df['qc_flag'].str.strip().str.upper()

microBio_df

In [ ]:
# Fill the patients without organism detected with 'NO_ORG_DETECTED'
microBio_df['org_name'] = microBio_df['org_name'].fillna('NO_ORG_DETECTED')
microBio_df

In [ ]:
# Filling missing values in dilution columns
microBio_df['dilution_comparison'] = microBio_df['dilution_comparison'].fillna('=')
microBio_df['dilution_value'] = microBio_df['dilution_value'].fillna(-1)
microBio_df['dilution_text'] = microBio_df['dilution_text'].fillna('-1')
microBio_df

In [ ]:
# Fill ab_name missing values with 'NO_AB_TESTED' if organism has been detected but ab_name is missing
mask_no_ab = (microBio_df['org_name'] != 'NO_ORG_DETECTED') & (microBio_df['ab_name'].isna())
microBio_df.loc[mask_no_ab, 'ab_name'] = 'NO_AB_TESTED'
microBio_df

In [ ]:
# FIll the ab_name missing values with 'NO_AB_DETECTED' where no organism has been detected
microBio_df.loc[microBio_df['org_name'] == 'NO_ORG_DETECTED', 'ab_name'] = 'NO_AB_DETECTED'
microBio_df

In [ ]:
# Fill the interpretation missing values with 'NO_INTERPRETATION' where ab_name is 'NO_AB_TESTED', 'NO_AB_FOUND', 'NO_AB_DETECTED'
mask_no_interp = microBio_df['ab_name'].isin(['NO_AB_TESTED', 'NO_AB_DETECTED']) & (microBio_df['interpretation'].isna())
microBio_df.loc[mask_no_interp, 'interpretation'] = 'NO_INTERPRETATION'
microBio_df

In [ ]:
#Save the new CSV
print(microBio_df['subject_id'].nunique())
microBio_df.to_csv("../../data/processed/microbio.csv", index=False)